In [1]:

import numpy as np
import math

# Job processing times and priority weights
processing_time = np.array([5, 2, 8, 3, 6, 4])
weight = np.array([3, 5, 2, 4, 1, 3])

n = len(processing_time)

# Cuckoo Search parameters
n_nests = 10
Pa = 0.25
max_iter = 100
alpha = 0.1
beta = 1.5

# Fitness function: Total weighted completion time
def fitness(position):
    order = np.argsort(position)

    completion_time = 0
    total_cost = 0

    for job in order:
        completion_time += processing_time[job]
        total_cost += weight[job] * completion_time

    return total_cost

# Generate Levy flight steps using Mantegna's algorithm
def levy_flight(size):
    sigma = (
        math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
        / (math.gamma((1 + beta) / 2) * beta *
           2 ** ((beta - 1) / 2))
    ) ** (1 / beta)

    u = np.random.normal(0, sigma, size)
    v = np.random.normal(0, 1, size)

    step = u / (np.abs(v) ** (1 / beta))
    return step

# Step 1: Initialize host nests
nests = np.random.rand(n_nests, n)
scores = np.array([fitness(x) for x in nests])

# Step 2: Find the initial best nest
best_index = np.argmin(scores)
best_nest = nests[best_index].copy()
best_score = scores[best_index]

# Step 3: Main Cuckoo Search loop
for iteration in range(max_iter):

    for i in range(n_nests):

        # Generate a new cuckoo using Levy flight
        new_solution = nests[i] + alpha * levy_flight(n)
        new_solution = np.clip(new_solution, 0, 1)

        new_score = fitness(new_solution)

        # Select a random host nest
        j = np.random.randint(n_nests)

        # Replace if the new solution is better
        if new_score < scores[j]:
            nests[j] = new_solution
            scores[j] = new_score

    # Step 4: Abandon a fraction of worst nests
    abandon_count = max(1, int(Pa * n_nests))

    worst_indices = np.argsort(scores)[-abandon_count:]

    for j in worst_indices:
        nests[j] = np.random.rand(n)
        scores[j] = fitness(nests[j])

    # Step 5: Update the best solution
    best_index = np.argmin(scores)

    if scores[best_index] < best_score:
        best_score = scores[best_index]
        best_nest = nests[best_index].copy()

# Step 6: Display the result
best_order = np.argsort(best_nest)

print("Best Job Scheduling Order:")

for job in best_order:
    print("Job J" + str(job + 1), end=" -> ")

print("\nMinimum Weighted Completion Time:", best_score)



Best Job Scheduling Order:
Job J2 -> Job J4 -> Job J6 -> Job J1 -> Job J3 -> Job J5 -> 
Minimum Weighted Completion Time: 171
